# Sheet 2 — Worked solutions

Arrays, plotting and benchmarking · DM587

Run from top to bottom with Python 3, NumPy, Matplotlib and pandas. Keep `benchmark.py`, `odense-weather.csv` and `DATA-SOURCE.md` in the same folder as this notebook. No Assignment 1 files are needed.

Each subquestion is labelled below. Outputs are saved in this version; the companion no-output version has exactly the same explanations and code. Random unseeded draws and benchmark timings will differ between runs. The code uses basic Python 3 syntax, but you need package versions supported by your Python installation.

Running the notebook writes `reciprocal.png` and `benchmark-results.csv` in its working folder. All benchmark checks use ordinary comparisons and printed messages; no assertions are required.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

for filename in ["benchmark.py", "odense-weather.csv"]:
    if not Path(filename).is_file():
        raise FileNotFoundError("Keep this file beside the notebook: " + filename)

plt.rcParams.update({"figure.figsize": (8, 4), "font.size": 11})
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


## Question 1 — A discontinuous function
### (a) One grid
The function is undefined at $x=1$. The 200-point grid below excludes 1, yet a single plotting call still joins points from opposite sides of the discontinuity. This creates a misleading, steep connecting segment.


In [ ]:
x_whole = np.linspace(-2, 6, 200)
print("Grid contains 1:", bool(np.any(x_whole == 1)))
fig, ax = plt.subplots()
ax.plot(x_whole, 1.0 / (x_whole - 1), color="magenta")
ax.set(xlim=(-2, 6), ylim=(-6, 6), xlabel="x", ylabel="f(x)",
       title="One plotting call joins across the discontinuity")
ax.axvline(1, color="grey", linestyle=":", linewidth=1)
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


### (b)–(c) Separate branches and styling
Plot each side separately so that Matplotlib never draws a line across the gap. The two sampled branches approach the vertical asymptote; neither includes $x=1$. The y-limits deliberately clip large magnitudes near the asymptote. The saved image is the corrected plot.


In [ ]:
def plot_reciprocal():
    left = np.linspace(-2, 0.99, 300)
    right = np.linspace(1.01, 6, 300)
    fig, ax = plt.subplots()
    ax.plot(left, 1.0 / (left - 1), "m--", linewidth=4)
    ax.plot(right, 1.0 / (right - 1), "m--", linewidth=4)
    ax.set(xlim=(-2, 6), ylim=(-6, 6), xlabel="x", ylabel="f(x)",
           title="f(x) = 1 / (x - 1)")
    ax.axvline(1, color="grey", linestyle=":", linewidth=1)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig("reciprocal.png", dpi=150)
    return fig, ax

fig, ax = plot_reciprocal()
print("Saved reciprocal.png")
plt.show()


## Question 2 — Subplots
### (a)–(b) Shared limits, titles and styles
`axs` is a 2-by-2 array of plotting panels. Shared limits allow a fair visual comparison. The overall title describes the figure; each panel names its own function.


In [ ]:
x = np.linspace(0, 2 * np.pi, 400)
fig, axs = plt.subplots(2, 2, figsize=(9, 6), sharex=True, sharey=True)
axs[0, 0].plot(x, np.sin(x), "g-")
axs[0, 1].plot(x, np.sin(2 * x), "r--")
axs[1, 0].plot(x, 2 * np.sin(x), "b--")
axs[1, 1].plot(x, 2 * np.sin(2 * x), "m:")
titles = [["sin(x)", "sin(2x)"], ["2 sin(x)", "2 sin(2x)"]]
for i in range(2):
    for j in range(2):
        axs[i, j].set(xlim=(0, 2 * np.pi), ylim=(-2, 2),
                      title=titles[i][j], xlabel="x (radians)", ylabel="y")
        axs[i, j].grid(alpha=0.2)
fig.suptitle("Amplitude and frequency")
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()


### (c) Interpretation
Multiplying the output by 2 doubles the amplitude from 1 to 2. Replacing $x$ by $2x$ doubles the frequency: two oscillations fit in $[0,2\pi]$, and the period changes from $2\pi$ to $\pi$. These are separate changes.


## Question 3 — Matrix powers
Use matrix products for the powers. `A ** 2` would square individual entries and solve a different problem. Reuse the calculated square rather than calculating it twice.


In [ ]:
def matrix_polynomial(A):
    A2 = np.dot(A, A)
    A3 = np.dot(A2, A)
    return -A3 + 9 * A2 - 15 * A

A = np.array([[3, 1, 4], [1, 5, 9], [-5, 3, 1]])
polynomial_result = matrix_polynomial(A)
print(polynomial_result)
print("Zero matrix:", np.array_equal(polynomial_result, np.zeros((3, 3))))


The result is the 3-by-3 zero matrix, not $A$. For this integer example the arithmetic is exact at these sizes. This illustrates the Cayley-Hamilton relation stated in the question; no proof is needed.


## Question 4 — Constructing matrices
### (a) Ones on and above the diagonal
`np.triu` keeps the upper triangle, including the main diagonal by default.


In [ ]:
ones7 = np.ones((7, 7))
upper = np.triu(ones7)
print("A:")
print(upper)
print("Shape:", upper.shape)


### (b) Fives strictly above the diagonal
`np.triu(ones7, k=1)` excludes the main diagonal. Subtracting the lower triangle of ones supplies minus ones on and below that diagonal. The two regions do not overlap.


In [ ]:
mixed = 5 * np.triu(ones7, k=1) - np.tril(ones7)
print("B:")
print(mixed)
print("Shape:", mixed.shape)


### (c) Return the product as `int64`
Both arrays are 7-by-7, so both successive matrix products are defined. The construction starts with floating-point arrays; converting at the end gives the requested integer type.


In [ ]:
def triangular_product():
    ones = np.ones((7, 7))
    A = np.triu(ones)
    B = 5 * np.triu(ones, k=1) - np.tril(ones)
    return np.dot(np.dot(A, B), A).astype(np.int64)

triangular_result = triangular_product()
print("ABA:")
print(triangular_result)
print("Shape:", triangular_result.shape)
print("Data type:", triangular_result.dtype)


## Question 5 — Array toolkit
### (a) Constructing arrays and inspecting types
Zeros, ones and the identity use floating-point values by default. The diagonal matrix below inherits integer values from its input. `linspace` gives exactly eleven positions including both endpoints; there are ten half-unit intervals from 0 to 5.


In [ ]:
zeros = np.zeros((3, 4))
ones = np.ones((3, 4))
identity = np.eye(4)
diagonal = np.diag(np.arange(1, 5))
integers = np.arange(5)
half_steps = np.linspace(0, 5, 11)
for name, array in [("Zeros", zeros), ("Ones", ones),
                    ("Identity", identity), ("Diagonal", diagonal),
                    ("Integers", integers), ("Half steps", half_steps)]:
    print(name)
    print(array)
    print("Shape:", array.shape, "dtype:", array.dtype)


### (b) Random arrays and endpoints
Uniform draws are in $[0,1)$: 0 is allowed and 1 is excluded. Integer draws use an excluded upper bound, so 11 allows integers 1 through 10. A fixed seed makes this demonstration repeatable in the same generator environment.


In [ ]:
rng = np.random.default_rng(587)
uniform_array = rng.random((3, 4))
integer_array = rng.integers(1, 11, size=(3, 4))
print("Uniform values:")
print(uniform_array)
print("Shape:", uniform_array.shape)
print("Integers 1 through 10:")
print(integer_array)
print("Shape:", integer_array.shape)


### (c) Indexing, shapes and data types
An integer index selects a single position in an axis; a slice retains that axis. A row or column selected with an integer index here is a one-dimensional array, rather than a row-shaped or column-shaped two-dimensional array. Default integer width may depend on the environment; inspect it rather than assuming it.


In [ ]:
C = np.arange(1, 10).reshape(3, 3)
print("C:")
print(C)
selections = [("Entry (0, 0)", C[0, 0]),
              ("Rows from the second", C[1:, :]),
              ("Second row", C[1, :]),
              ("Third column", C[:, 2])]
for name, value in selections:
    print(name)
    print(value)
    print("Shape:", np.shape(value))
print("C dtype:", C.dtype)
C_float = C.astype(np.float64)
print("Converted dtype:", C_float.dtype)
print(C_float)


### (d) Flattening, shared data and explicit copies
For each method, start with a fresh contiguous array. Both flattened arrays below share memory with their source, so their changes are visible in that source. An explicit `.copy()` creates independent numerical storage. Flattening is not a promise to copy; the behaviour can differ for non-contiguous arrays.


In [ ]:
for method in ["reshape", "ravel"]:
    original = np.arange(1, 10).reshape(3, 3)
    if method == "reshape":
        flat = original.reshape(-1)
    else:
        flat = original.ravel()
    print(method, "shares memory:", np.shares_memory(original, flat))
    flat[0] = 99
    print("Original after changing the view:")
    print(original)

    fresh = np.arange(1, 10).reshape(3, 3)
    if method == "reshape":
        independent = fresh.reshape(-1).copy()
    else:
        independent = fresh.ravel().copy()
    independent[0] = 99
    print("Independent flattened copy:", independent)
    print("Fresh source remains unchanged:")
    print(fresh)


### (e) Stacking and inserting a row
Vertical stacking combines rows: $(3,4)+(3,4)$ becomes $(6,4)$. Horizontal stacking combines columns and gives $(3,8)$. A new column for a 4-by-4 identity must be shaped $(4,1)$; a one-dimensional array of shape $(4,)$ is not interchangeable here. Adding a row to the 3-by-3 array gives four rows.


In [ ]:
X = np.arange(12).reshape(3, 4)
Y = X + 100
vertical = np.vstack((X, Y))
horizontal = np.hstack((X, Y))
print("Vertical:", vertical.shape)
print(vertical)
print("Horizontal:", horizontal.shape)
print(horizontal)
identity_with_column = np.hstack((np.eye(4), np.ones((4, 1))))
print("Identity with an extra ones column:", identity_with_column.shape)
print(identity_with_column)
C = np.arange(1, 10).reshape(3, 3)
inserted = np.insert(C, 1, [10, 11, 12], axis=0)
print("Inserted row:", inserted.shape)
print(inserted)


### (f) Transpose, two kinds of multiplication, and rank
`np.dot(C, C.T)` forms row–row dot products. `C * C` squares entries separately. The fresh array has rank 2: its rows are dependent; twice its second row minus its first equals its third.


In [ ]:
C = np.arange(1, 10).reshape(3, 3)
print("Transpose:")
print(C.T)
gram = np.dot(C, C.T)
print("C times C transpose:")
print(gram)
print("Shape:", gram.shape)
print("Entrywise C * C:")
print(C * C)
print("Rank of C:", np.linalg.matrix_rank(C))


### (g) Invertible and singular matrices
A 2-by-2 matrix has an inverse only if its rank is 2. $P$ has determinant $-1/2$ and is invertible. The second row of $Q$ is $-2/3$ times its first, so its rank is 1 and it is singular. We do not attempt to invert $Q$: floating-point inversion near singularity can fail or return meaningless large values.


In [ ]:
P = np.array([[-1.0, 1.5], [1.0, -1.0]])
Q = np.array([[-1.0, 1.5], [2.0 / 3, -1.0]])
for name, matrix in [("P", P), ("Q", Q)]:
    rank = np.linalg.matrix_rank(matrix)
    print(name, "rank:", rank)
    if rank == matrix.shape[0]:
        inverse = np.linalg.inv(matrix)
        print("Inverse:")
        print(inverse)
        print("Product:")
        print(np.dot(matrix, inverse))
        print("Close to identity:", np.allclose(np.dot(matrix, inverse), np.eye(2)))
    else:
        print("No inverse: rows are linearly dependent.")


## Question 6 — Seeds and reproducibility
### (a) Drawing twice from one generator
The generator advances its internal state after each draw; using it twice does not restart it.


In [ ]:
rng = np.random.default_rng(587)
first = rng.integers(0, 2, size=20)
second = rng.integers(0, 2, size=20)
print("First:", first)
print("Second:", second)
print("Identical:", np.array_equal(first, second))


### (b) Recreating the seeded generator
Recreating a generator with the same seed restarts the sequence. The same draw shape and order of calls reproduce `first`, rather than `second`.


In [ ]:
restarted = np.random.default_rng(587)
repeated = restarted.integers(0, 2, size=20)
print("Restarted:", repeated)
print("Matches first:", np.array_equal(first, repeated))


### (c) Different seed or no explicit seed
A different seed normally produces a different sequence in this example; it is not a guarantee that every possible draw differs. With no seed supplied, NumPy obtains initial entropy from the environment and this run is not reproducible from our code alone. For a reproducible experiment, record a fixed seed, the code and package environment.


In [ ]:
different = np.random.default_rng(588).integers(0, 2, size=20)
unseeded = np.random.default_rng().integers(0, 2, size=20)
print("Different seed:", different)
print("Matches first:", np.array_equal(first, different))
print("No supplied seed:", unseeded)


### (d) Restarting inside a loop
Restarting with the same seed at every iteration repeatedly selects the first value of the same stream. Create the generator once outside the loop to draw successive values. A bulk draw is equivalent to successive draws for the example below.


In [ ]:
restarted_each_time = []
for i in range(20):
    rng = np.random.default_rng(587)
    restarted_each_time.append(int(rng.integers(0, 2)))

rng = np.random.default_rng(587)
successive = []
for i in range(20):
    successive.append(int(rng.integers(0, 2)))
print("Reset in every iteration:", restarted_each_time)
print("One generator:", successive)
print("One generator matches first:", np.array_equal(successive, first))


## Question 7 — Rain and temperature range
### (a) Compare two groups within July 2024
Load the supplied CSV, calculate daily maximum minus minimum, and select one month. The wet-day definition is at least 1 mm; the other group may include light rain below 1 mm. Comparing days within the same month reduces the broad seasonal contrast present in a whole-year scatter plot, but does not control all other factors.


In [ ]:
weather = pd.read_csv("odense-weather.csv", parse_dates=["date"])
weather = weather.set_index("date").sort_index()
weather["temp_range"] = weather["temp_max"] - weather["temp_min"]
july = weather.loc["2024-07-01":"2024-07-31"]
wet_days = july.loc[july["rain_mm"] >= 1]
below_threshold = july.loc[july["rain_mm"] < 1]
comparison = pd.DataFrame({
    "days": [len(below_threshold), len(wet_days)],
    "mean_range_C": [below_threshold["temp_range"].mean(),
                     wet_days["temp_range"].mean()]},
    index=["Below 1 mm", "At least 1 mm"])
print(comparison.round(2).to_string())


There are 14 days below 1 mm and 17 wet days. Their mean ranges are 9.80 and about 7.71 degrees C, respectively. The wet group therefore has a smaller mean by about 2.09 degrees C. These are group summaries, not a claim that every wet day has a smaller range than every other day.


### (b) Scatter plot and interpretation
Each point is one July day. The vertical line separates the groups used in part (a); the y-axis shows the derived temperature range rather than minimum or maximum temperature itself.


In [ ]:
fig, ax = plt.subplots()
ax.scatter(july["rain_mm"], july["temp_range"],
           color="#006D77", s=45, alpha=0.8)
ax.axvline(1, color="#D56A35", linestyle="--", label="Wet-day threshold: 1 mm")
ax.set(title="Rain and daily temperature range, July 2024",
       xlabel="Daily rainfall (mm)", ylabel="Daily temperature range (degrees C)")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


The scatter shows substantial variation and overlap, so a difference between group means is not a reliable rule for individual days. Rainy and less rainy days also differ in factors such as cloud cover and weather systems. This observational comparison does not establish that rainfall alone causes the temperature-range difference.


### (c) Fraction of wet days by month
Count both wet days and all recorded days in each month. Dividing within each month accounts for different month lengths. The extract has complete dates for 2024, including 29 February. A boolean Series can be summed because `True` contributes 1 and `False` contributes 0.


In [ ]:
year = weather.loc["2024"]
wet_indicator = year["rain_mm"] >= 1
wet_counts = wet_indicator.groupby(year.index.month).sum()
day_counts = wet_indicator.groupby(year.index.month).count()
wet_fraction = wet_counts / day_counts
monthly_summary = pd.DataFrame({"wet_days": wet_counts,
                                "recorded_days": day_counts,
                                "wet_fraction": wet_fraction})
print(monthly_summary.round(3).to_string())
print("Largest wet-day fraction: month", int(wet_fraction.idxmax()))
fig, ax = plt.subplots()
ax.bar(wet_fraction.index, wet_fraction.values, color="#006D77")
ax.set_xticks(np.arange(1, 13))
ax.set(ylim=(0, 1), title="Fraction of days with at least 1 mm rain, 2024",
       xlabel="Calendar month", ylabel="Fraction of recorded days")
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()


April (month 4) has the largest fraction: 20 of its 30 days, or about 0.667. Raw counts ignore unequal month lengths; total rainfall measures the amount of rain rather than how often the threshold is reached. A month with a few very heavy rain events can have a high total without a high wet-day fraction.

**Data source:** Copernicus ERA5 reanalysis via Open-Meteo, CC BY 4.0. These are gridded estimates near Odense, not observations from a named station. See `DATA-SOURCE.md` for provenance.


## Question 8 — Benchmarking matrix multiplication
Important preparation for Assignment 1.

### (a) Supplied functions and a small correctness check
`matmul_loop` computes the product. `mult_loop` and `mult_numpy` are the timing wrappers. Their returned numbers are times, not matrix products; use the raw calculation for correctness checks. The `(2, 3)` times `(3, 2)` product has shape `(2, 2)`. Entry `(0, 1)` is $1\cdot0+2\cdot1+3\cdot1=5$.

The following import does not run the benchmark's `main()` because its test is under `if __name__ == "__main__":`.


In [ ]:
from benchmark import matmul_loop

A = np.array([[1, 2, 3], [4, 5, 6]], dtype=float)
B = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
loop_result = matmul_loop(A, B)
numpy_result = np.dot(A, B)
print("Loop product:")
print(loop_result)
print("Shape:", loop_result.shape)
print("Entry (0, 1):", loop_result[0, 1])
print("Agrees with NumPy:", np.allclose(loop_result, numpy_result))


### (b) Run the supplied script
In a terminal in this folder, use `python3 benchmark.py`. Here `runpy` executes the same local file as a program from the notebook, including its main block. The supplied settings use sizes 5, 10, 20 and 40; no implementation changes are needed. A fresh benchmark CSV is written on every run.


In [ ]:
import runpy
benchmark_run = runpy.run_path("benchmark.py", run_name="__main__")


### (c) Read the timing units
Each printed figure is the best **batch total**, measured over three calls, among three batches. The CSV divides that total by three, so its timings are **seconds per call**. Directly comparing a batch total with a per-call time would introduce a factor-of-three error.

Keeping allocation, data conversion, correctness checks, printing and CSV writing outside the timed region focuses the comparison on multiplication. Both methods receive the same input values, shapes and floating-point data type. Repeated batches reduce the influence of transient delays; they do not remove all measurement noise.


In [ ]:
timings = pd.read_csv("benchmark-results.csv")
print("Per-call times and speed ratios:")
print(timings.to_string(index=False))
print("Call count per batch:", benchmark_run["LOOPS"])
print("Number of batches:", benchmark_run["REPEATS"])


### (d) Ordinary and log-log plots
A speed ratio of 4 means the loop version took four times as long as NumPy, or NumPy took one quarter of the time. The horizontal variable is width $n$. On ordinary axes, the much smaller NumPy times may be hard to see; log-log axes let us compare changes over orders of magnitude. Both series must contain positive times for logarithmic axes.


In [ ]:
if (timings[["loop_seconds", "numpy_seconds"]] <= 0).any().any():
    raise ValueError("A timing rounded to zero; increase LOOPS and rerun the benchmark")
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax in axs:
    ax.plot(timings["n"], timings["loop_seconds"], "o-", label="Python loops")
    ax.plot(timings["n"], timings["numpy_seconds"], "o-", label="NumPy")
    ax.set(xlabel="Square matrix width n", ylabel="Seconds per call")
    ax.legend()
    ax.grid(alpha=0.2)
axs[0].set_title("Ordinary axes")
axs[1].set_title("Log-log axes")
axs[1].set_xscale("log")
axs[1].set_yscale("log")
fig.suptitle("Matrix multiplication on this machine")
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()


### (e) A second timing method
Use a fixed size, created before the timed calls. Both comparisons below use the fastest of three batches and divide by three calls. This makes the summary convention comparable with the supplied timer, although clocks and measurement overheads differ.

In IPython/Jupyter, `%timeit -n 3 -r 3 matmul_loop(A, B)` is an alternative interactive command. It is not ordinary Python syntax. Its usual mean/standard-deviation summary differs from the minimum used here. We use the standard-library form below so the cell is also valid Python code.


In [ ]:
import timeit
rng_timing = np.random.default_rng(587)
A_timing = rng_timing.random((40, 40))
B_timing = rng_timing.random((40, 40))
loop_batches = timeit.repeat(lambda: matmul_loop(A_timing, B_timing),
                            number=3, repeat=3)
numpy_batches = timeit.repeat(lambda: np.dot(A_timing, B_timing),
                             number=3, repeat=3)
loop_per_call = min(loop_batches) / 3
numpy_per_call = min(numpy_batches) / 3
print("timeit loop seconds per call:", loop_per_call)
print("timeit NumPy seconds per call:", numpy_per_call)
print("timeit speed ratio:", loop_per_call / numpy_per_call)
print("NumPy faster in this run:", numpy_per_call < loop_per_call)


### (f) Growth, slopes and limitations
The loop performs $n^2$ output calculations, each summing $n$ products. This explains why cubic work is a useful theoretical reference for this particular implementation. On a log-log plot a pure $c n^p$ law has slope $p$.

The following local slopes describe only adjacent measured points. They are not complexity estimates with a statistical guarantee. Tiny matrices can be dominated by call overhead and timer resolution. NumPy's compiled routines and hardware behaviour differ from Python loops, so measured slopes and speed ratios need not be constant. Compare the broad pattern across repeated runs rather than treating any single timing as definitive.


In [ ]:
widths = timings["n"].to_numpy(dtype=float)
loop_values = timings["loop_seconds"].to_numpy()
numpy_values = timings["numpy_seconds"].to_numpy()
loop_slopes = np.diff(np.log(loop_values)) / np.diff(np.log(widths))
numpy_slopes = np.diff(np.log(numpy_values)) / np.diff(np.log(widths))
print("Size intervals:")
for i in range(len(widths) - 1):
    print(int(widths[i]), "to", int(widths[i + 1]),
          "loop slope:", round(float(loop_slopes[i]), 2),
          "NumPy slope:", round(float(numpy_slopes[i]), 2))
print("Speed ratios:", timings["speed_ratio"].round(2).tolist())


The saved output is one measurement on the machine used to prepare this notebook. Your numerical timing values will differ. If the NumPy timings are too short to measure reliably, increase the calls per batch and repeat; compare **per-call** averages, not batch totals with different call counts. No conclusion about correctness follows merely from being fast.
